In [ ]:
# 1) Install required packages (run once)
!pip install -q torch-scatter torch-sparse torch-cluster torch-spline-conv torch-geometric==2.7.0 -f https://data.pyg.org/whl/torch-2.1.0+cu118.html
!pip install -q transformers
!pip install -q pandas scikit-learn tqdm

# 2) Imports
import os, math, random, time
import numpy as np
import pandas as pd
from collections import Counter
from tqdm import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

from transformers import AutoTokenizer, AutoModel
from sklearn.model_selection import train_test_split
from sklearn.metrics import precision_recall_fscore_support

import torch_geometric.nn as pyg_nn
from torch_geometric.nn import GATv2Conv
import warnings
warnings.filterwarnings("ignore")

# 3) Config / Hyperparams (tune as required)
DATA_PATH = "/kaggle/input/btp-dataset/processed2_dataset.csv"   # <--- your uploaded file path
PRETRAINED_MODEL = "distilbert-base-uncased"
BATCH_SIZE = 16
MAX_LEN = 128
LABEL_DIM = 256           # d in paper (label feature dim)
TEXT_PROJ_DIM = LABEL_DIM # project text -> same dim as label embeddings
NUM_HEADS = 4
LR = 3e-5
WEIGHT_DECAY = 1e-2
NUM_EPOCHS = 25
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
SEED = 42

# reproducibility
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available(): torch.cuda.manual_seed(SEED)

# 4) Load dataframe (assumes your CSV has 'input_text' and label columns as binary 0/1 columns)
df = pd.read_csv(DATA_PATH)
print("Loaded", len(df), "rows. Columns:", df.columns.tolist())

# Identify text column and label columns
TEXT_COL = "input_text"
# labels assumed to be remaining columns (except input_text). Adjust if needed.
label_cols = [c for c in df.columns if c != TEXT_COL]
num_labels = len(label_cols)
print("Detected", num_labels, "labels.")

In [ ]:

# 5) Prepare label arrays and train/val/test split
Y = df[label_cols].values.astype(np.float32)   # shape (N, C)
texts = df[TEXT_COL].astype(str).tolist()

# split
train_idx, test_idx = train_test_split(range(len(df)), test_size=0.10, random_state=SEED, shuffle=True)
train_idx, val_idx = train_test_split(train_idx, test_size=0.1111, random_state=SEED, shuffle=True) 
# approximately 80/10/10 split
print("Splits sizes:", len(train_idx), len(val_idx), len(test_idx))

# 6) Tokenize with HuggingFace (DistilBERT tokenizer)
tokenizer = AutoTokenizer.from_pretrained(PRETRAINED_MODEL)
# batch tokenization (returns torch tensors)
encodings = tokenizer(texts, padding='max_length', truncation=True, max_length=MAX_LEN, return_tensors='pt')

input_ids_all = encodings['input_ids']   # (N, MAX_LEN)
attention_mask_all = encodings['attention_mask']

# 7) Build weighted label co-occurrence adjacency (counts -> normalized adjacency)
Y_train = Y[train_idx]   # (n_train, C)
co_counts = np.dot(Y_train.T, Y_train).astype(np.float32)  # co-occurrence counts (C x C)
# remove self-counts then add later if desired
np.fill_diagonal(co_counts, 0.0)

# Optionally use PMI; here we keep normalized co-occurrence (symmetric normalization)
deg = co_counts.sum(axis=1)  # degree
# avoid divide by zero
deg_safe = deg + 1e-8
D_inv_sqrt = np.diag(1.0 / np.sqrt(deg_safe))
adj_norm = D_inv_sqrt @ co_counts @ D_inv_sqrt  # symmetric normalized adjacency
# keep self-loop weight to 1.0 on diagonal (helps GNN stability)
np.fill_diagonal(adj_norm, 1.0)

# Build edge_index and (optionally) edge_weight arrays from adj_norm
src, dst = np.nonzero(adj_norm)
edge_index = torch.tensor(np.vstack([src, dst]), dtype=torch.long).contiguous().to(DEVICE)  # [2, E]
edge_weight = torch.tensor(adj_norm[src, dst], dtype=torch.float32).to(DEVICE)           # [E]

print("Label adjacency edges:", edge_index.size(1))

# 8) Dataset and DataLoader wrappers (use pre-tokenized tensors for speed)
class TextDataset(Dataset):
    def __init__(self, indices):
        self.indices = indices
    def __len__(self): return len(self.indices)
    def __getitem__(self, idx):
        i = self.indices[idx]
        return {
            "input_ids": input_ids_all[i],           # torch tensor
            "attention_mask": attention_mask_all[i],
            "labels": torch.tensor(Y[i], dtype=torch.float32)
        }

train_ds = TextDataset(train_idx)
val_ds   = TextDataset(val_idx)
test_ds  = TextDataset(test_idx)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True)
val_loader   = DataLoader(val_ds, batch_size=BATCH_SIZE)
test_loader  = DataLoader(test_ds, batch_size=BATCH_SIZE)

In [ ]:
# 9) Model: DistilBERT encoder -> project to label_dim -> label GNN (GATv2) -> dot-product fusion
class PaperModel(nn.Module):
    def __init__(self, pretrained_name, label_dim, num_labels, num_heads, edge_index, edge_weight=None, device='cpu'):
        super().__init__()
        # text encoder
        self.bert = AutoModel.from_pretrained(pretrained_name)
        bert_hidden = self.bert.config.hidden_size  # e.g. 768
        # project BERT pooled vector -> label_dim
        self.text_proj = nn.Sequential(
            nn.Linear(bert_hidden, label_dim),
            nn.ReLU(),
            nn.Dropout(0.2)
        )
        # label initial embeddings (learnable)
        self.label_emb = nn.Parameter(torch.randn(num_labels, label_dim) * 0.1)
        # incorporate adjacency (we will pre-multiply label_emb by adj_norm externally in forward)
        # GAT on labels
        self.gat = GATv2Conv(in_channels=label_dim, out_channels=label_dim, heads=num_heads, concat=False, dropout=0.2)
        # final projection for label features
        self.label_proj = nn.Sequential(nn.Linear(label_dim, label_dim), nn.ReLU(), nn.Dropout(0.2))
        self.edge_index = edge_index  # stored device tensors
        self.edge_weight = edge_weight
    def forward(self, input_ids, attention_mask, adj_norm_matrix=None):
        # Text encoding (DistilBERT)
        bert_out = self.bert(input_ids=input_ids, attention_mask=attention_mask, return_dict=True)
        # mean pooling over token outputs using attention mask
        last_hidden = bert_out.last_hidden_state  # (B, L, H)
        mask = attention_mask.unsqueeze(-1).to(last_hidden.dtype)
        pooled = (last_hidden * mask).sum(dim=1) / (mask.sum(dim=1).clamp(min=1e-9))  # (B, H)
        text_vec = self.text_proj(pooled)  # (B, label_dim)

        # Build label node features: we use adj_norm to mix initial label embeddings -> then GAT
        # adj_norm_matrix is numpy array (C x C) normalized; convert to tensor on same device as label_emb
        if adj_norm_matrix is not None:
            device = self.label_emb.device
            A = torch.tensor(adj_norm_matrix, dtype=torch.float32, device=device)  # (C, C)
            # Weighted aggregate of neighbor embeddings as initial features
            x0 = torch.matmul(A, self.label_emb)   # (C, label_dim)
        else:
            x0 = self.label_emb  # (C, label_dim)
        # run GAT: GATv2Conv expects x [C, D] and edge_index [2, E]
        x = self.gat(x0, self.edge_index)  # (C, label_dim)
        x = self.label_proj(x)             # (C, label_dim), final H

        # Fusion: dot product between text_vec (B, d) and H (C, d) -> logits (B, C)
        logits = torch.matmul(text_vec, x.t())
        return logits, x, text_vec

# 10) Instantiate model, optimizer, loss (with pos_weight)
model = PaperModel(pretrained_name=PRETRAINED_MODEL, label_dim=LABEL_DIM, num_labels=num_labels,
                   num_heads=NUM_HEADS, edge_index=edge_index, edge_weight=None, device=DEVICE).to(DEVICE)

# compute pos_weight from training labels
label_counts = Y_train.sum(axis=0)  # length C
pos_weight = torch.tensor((len(Y_train) - label_counts) / (label_counts + 1e-9), dtype=torch.float32).to(DEVICE)
criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)

optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=2, verbose=True)

In [ ]:

# 11) Utility functions: micro metrics and threshold tuning
def micro_metrics_from_counts(TP, FP, FN):
    prec = TP / (TP + FP) if (TP + FP) > 0 else 0.0
    rec  = TP / (TP + FN) if (TP + FN) > 0 else 0.0
    f1   = 2 * prec * rec / (prec + rec) if (prec + rec) > 0 else 0.0
    return prec, rec, f1

@torch.no_grad()
def evaluate_loader(model, loader, threshold=0.5, adj_norm=None):
    model.eval()
    TP = FP = FN = 0
    all_probs = []
    all_targets = []
    for batch in loader:
        input_ids = batch['input_ids'].to(DEVICE)
        attention_mask = batch['attention_mask'].to(DEVICE)
        labels = batch['labels'].to(DEVICE)
        logits, _, _ = model(input_ids, attention_mask, adj_norm_matrix=adj_norm)
        probs = torch.sigmoid(logits)
        all_probs.append(probs.cpu().numpy())
        all_targets.append(labels.cpu().numpy())
        preds = (probs > threshold).float()
        TP += ((preds == 1) & (labels == 1)).sum().item()
        FP += ((preds == 1) & (labels == 0)).sum().item()
        FN += ((preds == 0) & (labels == 1)).sum().item()
    all_probs = np.vstack(all_probs)
    all_targets = np.vstack(all_targets)
    prec, rec, f1 = micro_metrics_from_counts(TP, FP, FN)
    return prec, rec, f1, all_probs, all_targets

# 12) Training loop (with gradient clipping)
best_val_f1 = 0.0
best_state = None
adj_norm_np = adj_norm  # numpy (C x C) computed earlier

for epoch in range(1, NUM_EPOCHS+1):
    model.train()
    epoch_loss = 0.0
    for batch in train_loader:
        input_ids = batch['input_ids'].to(DEVICE)
        attention_mask = batch['attention_mask'].to(DEVICE)
        labels = batch['labels'].to(DEVICE)

        optimizer.zero_grad()
        logits, _, _ = model(input_ids, attention_mask, adj_norm_matrix=adj_norm_np)
        loss = criterion(logits, labels)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()

        epoch_loss += loss.item() * input_ids.size(0)
    avg_loss = epoch_loss / len(train_idx)
    # validation
    val_prec, val_rec, val_f1, _, _ = evaluate_loader(model, val_loader, threshold=0.5, adj_norm=adj_norm_np)
    print(f"Epoch {epoch}: train_loss={avg_loss:.4f}  Val P={val_prec:.4f} R={val_rec:.4f} F1={val_f1:.4f}")
    # scheduler uses validation loss
    scheduler.step(avg_loss)

    if val_f1 > best_val_f1:
        best_val_f1 = val_f1
        best_state = model.state_dict()
        torch.save(best_state, "best_model_paper_style.pt")
        print("Saved new best model.")

print("Best val F1:", best_val_f1)


In [ ]:
# 13) After training: tune per-label thresholds on validation set (maximize micro-F1)
# Get validation probabilities and targets
_, _, _, val_probs, val_targets = evaluate_loader(model, val_loader, threshold=0.5, adj_norm=adj_norm_np)
C = num_labels
best_thresholds = np.ones(C) * 0.5
# brute-force search per label (coarse) to maximize micro-F1 when applied to all labels jointly
# We'll search thresholds in [0.1..0.9] per label independently (fast but not globally optimal)
for j in range(C):
    best_t = 0.5
    best_f = -1.0
    for t in np.linspace(0.1, 0.9, 17):
        preds = (val_probs > t).astype(int)
        TP = ((preds == 1) & (val_targets == 1)).sum()
        FP = ((preds == 1) & (val_targets == 0)).sum()
        FN = ((preds == 0) & (val_targets == 1)).sum()
        prec, rec, f1 = micro_metrics_from_counts(TP, FP, FN)
        if f1 > best_f:
            best_f = f1
            best_t = t
    best_thresholds[j] = best_t

print("Best per-label thresholds (first 20):", best_thresholds[:min(20,len(best_thresholds))])

# 14) Evaluate on test set using optimized thresholds
model.load_state_dict(best_state)  # best saved model
model.eval()
all_preds = []
all_targets = []
with torch.no_grad():
    for batch in test_loader:
        input_ids = batch['input_ids'].to(DEVICE)
        attention_mask = batch['attention_mask'].to(DEVICE)
        labels = batch['labels'].cpu().numpy()
        logits, _, _ = model(input_ids, attention_mask, adj_norm_matrix=adj_norm_np)
        probs = torch.sigmoid(logits).cpu().numpy()
        preds = (probs > best_thresholds).astype(int)
        all_preds.append(preds)
        all_targets.append(labels)

all_preds = np.vstack(all_preds)
all_targets = np.vstack(all_targets)
TP = ((all_preds == 1) & (all_targets == 1)).sum()
FP = ((all_preds == 1) & (all_targets == 0)).sum()
FN = ((all_preds == 0) & (all_targets == 1)).sum()
test_prec, test_rec, test_f1 = micro_metrics_from_counts(TP, FP, FN)
print(f"Test Micro Precision: {test_prec:.4f}, Recall: {test_rec:.4f}, F1: {test_f1:.4f}")

# 15) Extra: per-label detailed report (optional)
from sklearn.metrics import classification_report
print("Flattened classification report (micro-average shown above):")
print(classification_report(all_targets.ravel(), all_preds.ravel(), zero_division=0))

# Save thresholds for reproducibility
np.save("best_thresholds.npy", best_thresholds)
print("Done. Model + thresholds saved.")